In [8]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3' 
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [9]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install -q networkx psutil neuroCombat scikit-learn xgboost lightgbm seaborn statsmodels

In [10]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, json, hashlib, time, psutil, platform, subprocess, copy, random
from pathlib import Path
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.utils import dense_to_sparse, add_self_loops
import networkx as nx
import sklearn
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, roc_auc_score,
    roc_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve
from scipy.stats import wilcoxon
import joblib as jl

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

SEED = int(os.getenv('SEED', 42))
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# CHANGE 1: Guard deterministic algorithms in case of unsupported ops
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    warnings.warn("Deterministic algorithms not fully supported on this PyTorch/CUDA version.")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*60)
print("Workflow 2B - Population Graph Learning")
print("="*60)
print(f"Device       : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")
print(f"PyG          : {torch_geometric.__version__}")
print(f"Seed         : {SEED}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory   : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print("="*60)

# ── Paths ──
ROOT = Path(os.getenv('ADHD200_ROOT', '/mnt/ADHD200'))
W1_DIR = Path(os.getenv('W1_DIR', '/mnt/ADHD200_WORKFLOW1_RUN'))
W2_DIR = Path(os.getenv('W2_DIR', '/mnt/ADHD200_WORKFLOW2_RUN'))
W2A_DIR = W2_DIR / '06_results' / 'workflow2a'

required_artifacts = [
    W2A_DIR / "w2_pareto_front.csv",
    W2A_DIR / "w2_phase2_audit_all_reps.csv",
    W2A_DIR / "w2_site_signal_summary.csv",
]
for f in required_artifacts:
    assert f.exists(), f"Missing required W2A artifact: {f.name}. Please run W2A first."

RES_DIR = W2_DIR / '06_results' / 'workflow2b'
MODEL_DIR = RES_DIR / 'models'
FIG_DIR = RES_DIR / 'figures'
CACHE_DIR = RES_DIR / 'cache'   # CHANGE 9: Cache directory for intermediate results
for d in [RES_DIR, MODEL_DIR, FIG_DIR, RES_DIR / 'predictions', RES_DIR / 'embeddings',
          RES_DIR / 'confusion_matrices', FIG_DIR / 'reliability_diagrams', CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Results directory: {RES_DIR}\n")

# ── Environment Manifest ──
env_manifest = {
    'seed': SEED,
    'device': str(DEVICE),
    'torch_version': torch.__version__,
    'cuda_version': str(torch.version.cuda),
    'pyg_version': torch_geometric.__version__,
    'python_version': platform.python_version(),
    'platform': platform.platform(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'sklearn_version': sklearn.__version__,
    'timestamp': datetime.now().isoformat(),
    'hostname': platform.node(),
}
try:
    env_manifest['pip_freeze'] = subprocess.check_output(['pip', 'freeze']).decode().strip().split('\n')
except Exception:
    env_manifest['pip_freeze'] = []
try:
    env_manifest['git_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
except Exception:
    env_manifest['git_commit'] = 'unknown'

with open(RES_DIR / 'w2b_environment.json', 'w') as f:
    json.dump(env_manifest, f, indent=2)

print("Environment manifest saved.")

Workflow 2B - Population Graph Learning
Device       : cuda
PyTorch      : 2.5.1+cu121
PyG          : 2.8.0
Seed         : 42
GPU          : NVIDIA A100-SXM4-80GB
GPU Memory   : 79.3 GB
Results directory: /mnt/ADHD200_WORKFLOW2_RUN/06_results/workflow2b

Environment manifest saved.


fatal: not a git repository (or any of the parent directories): .git


In [11]:
start_t = time.time()

# Load core data
X_fc_norm = np.load(ROOT / '03_fc_matrices' / 'X_fc_norm.npy')
y_binary = np.load(ROOT / '03_fc_matrices' / 'y_binary.npy')
sites = np.load(ROOT / '03_fc_matrices' / 'sites.npy', allow_pickle=True).astype(str)
subjects = np.load(ROOT / '03_fc_matrices' / 'subjects.npy', allow_pickle=True).astype(str)
pheno_df = pd.read_parquet(W1_DIR / '04_features' / 'w1_pheno_features.parquet')
graph_df = pd.read_parquet(W1_DIR / '04_features' / 'w1_graph_features.parquet')

# Basic shape and length validation
n_sub, n_roi = X_fc_norm.shape[0], X_fc_norm.shape[1]
assert X_fc_norm.shape == (n_sub, n_roi, n_roi), f'FC shape mismatch: {X_fc_norm.shape}'
assert len(y_binary) == n_sub, 'y_binary length mismatch'
assert len(sites) == n_sub, 'sites length mismatch'
assert len(subjects) == n_sub, 'subjects length mismatch'
assert len(pheno_df) == n_sub, 'pheno_df length mismatch'
assert len(graph_df) == n_sub, 'graph_df length mismatch'

# Finite-value checks
assert np.isfinite(X_fc_norm).all(), 'Non-finite FC values found'
assert np.isfinite(y_binary).all(), 'Non-finite labels found'

# Symmetry validation
assert np.allclose(X_fc_norm, X_fc_norm.transpose(0, 2, 1), atol=1e-6), 'FC matrices are not symmetric'

# Diagonal statistics
diag = np.diagonal(X_fc_norm, axis1=1, axis2=2)
print(f'Diagonal mean: {diag.mean():.4f}, std: {diag.std():.4f}')

# Site distribution
site_counts = pd.Series(sites).value_counts().sort_index()
print('Site distribution:')
print(site_counts)

# Diagnosis per site
print('Diagnosis per site:')
print(pd.crosstab(sites, y_binary))

# Load W2A Pareto results (validation only)
pareto_df = pd.read_csv(W2A_DIR / 'w2_pareto_front.csv')
assert len(pareto_df) > 0, 'W2A Pareto front is empty'
selected_reps = pareto_df['Representation'].tolist()
print(f'W2A selected representations: {selected_reps}')

# Dataset statistics
adhd_frac = np.mean(y_binary)
print(f'Loaded: {n_sub} subjects, {n_roi} ROIs, {len(np.unique(sites))} sites')
print(f'Class balance: {adhd_frac:.3f} ADHD')

# Save dataset manifest
dataset_info = {
    'n_subjects': n_sub,
    'n_roi': n_roi,
    'n_sites': int(len(np.unique(sites))),
    'sites': list(np.unique(sites)),
    'adhd_prevalence': float(adhd_frac),
    'pheno_features': list(pheno_df.columns),
    'graph_features': list(graph_df.select_dtypes(include=[np.number]).columns),
    'selected_reps': selected_reps,
}
with open(RES_DIR / 'w2b_dataset_manifest.json', 'w') as f:
    json.dump(dataset_info, f, indent=2)

# Runtime logging
try:
    log_time('Data Loading & Validation', start_t)
except NameError:
    print(f'Data loading runtime: {time.time() - start_t:.2f} seconds')

Diagonal mean: -0.0416, std: 0.0223
Site distribution:
KKI           83
NYU           93
NeuroIMAGE    48
OHSU          79
Peking_1      85
Peking_2      67
Peking_3      42
Name: count, dtype: int64
Diagnosis per site:
col_0        0   1
row_0             
KKI         61  22
NYU         39  54
NeuroIMAGE  23  25
OHSU        42  37
Peking_1    61  24
Peking_2    32  35
Peking_3    23  19
W2A selected representations: ['ComBat FC', 'Raw GraphPheno', 'ComBat FCPheno', 'ComBat FCGraphPheno']
Loaded: 497 subjects, 190 ROIs, 7 sites
Class balance: 0.435 ADHD
Data loading runtime: 0.16 seconds


In [13]:
start_t = time.time()

# ── 1. Fixed threshold (hard‑coded from prior optimisation) ──
SELECTED_THRESHOLD = "top_10pct"
SELECTION_RATIONALE = (
    "Selected from subject-wise threshold optimisation based on "
    "graph connectivity, density, small-worldness, and downstream GCN performance."
)

# Define the threshold function (top positive 10%)
threshold_fn = lambda fc: np.where(
    fc >= np.percentile(fc[np.triu_indices_from(fc, k=1)], 90),
    fc, 0
)

# ── 2. Load or set node feature type ──
try:
    with open(RES_DIR / 'selected_node_features.json', 'r') as f:
        node_config = json.load(f)
    selected_node_feat = node_config['selected_node_features']
except FileNotFoundError:
    selected_node_feat = 'connectivity'   # default
    print("No node feature config found, using default 'connectivity'")

# ── 3. Define the graph construction function ──
def build_pyg_data(fc_matrix, y, site, subj, node_feat_type='connectivity'):
    """
    Build a PyG Data object from a single FC matrix.
    Uses the fixed threshold_fn (top_10pct) for edge sparsification.
    """
    assert np.isfinite(fc_matrix).all(), "FC matrix contains non-finite values"
    adj = threshold_fn(fc_matrix.copy())
    np.fill_diagonal(adj, 0)
    assert np.allclose(adj, adj.T), "Adjacency not symmetric"
    adj_t = torch.tensor(adj, dtype=torch.float32)
    edge_index, edge_attr = dense_to_sparse(adj_t)
    assert edge_index.shape[1] > 0, "Thresholded graph has no edges"
    edge_index, edge_attr_sl = add_self_loops(edge_index, edge_attr, num_nodes=fc_matrix.shape[0])
    if node_feat_type == 'identity':
        x = torch.eye(fc_matrix.shape[0], dtype=torch.float32)
    elif node_feat_type == 'connectivity':
        x = torch.tensor(fc_matrix, dtype=torch.float32)
    else:
        x = torch.eye(fc_matrix.shape[0], dtype=torch.float32)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr_sl,
                y=torch.tensor(y, dtype=torch.long),
                site=site, subject=subj)

# ── 4. Build dataset ──
print("Building dataset...")
dataset = [build_pyg_data(X_fc_norm[i], int(y_binary[i]), sites[i], subjects[i], selected_node_feat)
           for i in range(n_sub)]

# ── 5. Validation ──
assert len(dataset) == n_sub, f'Dataset size mismatch: {len(dataset)} vs {n_sub}'
assert all(d.y.item() in [0, 1] for d in dataset), "Invalid diagnosis labels"
assert len(np.unique([d.site for d in dataset])) == len(np.unique(sites)), "Site information not preserved"

sample = dataset[0]
assert sample.x.shape[0] == n_roi, f'Node count mismatch: {sample.x.shape[0]} vs {n_roi}'
assert sample.edge_index.shape[0] == 2, "Edge_index must be 2 x E"
assert sample.edge_attr.shape[0] == sample.edge_index.shape[1], "Edge_attr count does not match edge count"
assert sample.edge_index.shape[1] >= n_roi, "Self-loops may be missing"
assert torch.isfinite(sample.x).all(), "Non-finite node features"
assert torch.isfinite(sample.edge_attr).all(), "Non-finite edge attributes"
edge_counts = np.array([d.edge_index.shape[1] for d in dataset])
assert np.all(edge_counts > 0), "Some graphs contain no edges"

# ── 6. Summary ──
preproc_summary = {
    'N_Subjects': n_sub,
    'N_ROI': n_roi,
    'Node_Feature_Dim': int(sample.x.shape[1]),
    'Threshold': SELECTED_THRESHOLD,
    'Node_Feature_Type': selected_node_feat,
    'Self_Loops': True,
    'Edge_Weights': True,
    'Pooling': 'GlobalMeanPool',
    'Graph_Type': 'Functional Connectivity',
    'Mean_Edges': round(float(np.mean(edge_counts)), 2),
    'Std_Edges': round(float(np.std(edge_counts)), 2),
}
pd.DataFrame([preproc_summary]).to_csv(RES_DIR / 'graph_preprocessing_summary.csv', index=False)
print(json.dumps(preproc_summary, indent=2))

# Save the selected threshold info for reproducibility (optional)
with open(RES_DIR / 'selected_threshold.json', 'w') as f:
    json.dump({
        'selected_threshold': SELECTED_THRESHOLD,
        'rationale': SELECTION_RATIONALE,
        'method': 'hardcoded_from_prior_optimization'
    }, f, indent=2)

try:
    log_time('Graph Dataset Construction', start_t)
except NameError:
    print(f'Graph Dataset Construction runtime: {time.time() - start_t:.2f} seconds')

Building dataset...
{
  "N_Subjects": 497,
  "N_ROI": 190,
  "Node_Feature_Dim": 190,
  "Threshold": "top_10pct",
  "Node_Feature_Type": "connectivity",
  "Self_Loops": true,
  "Edge_Weights": true,
  "Pooling": "GlobalMeanPool",
  "Graph_Type": "Functional Connectivity",
  "Mean_Edges": 3782.0,
  "Std_Edges": 0.0
}
Graph Dataset Construction runtime: 25.15 seconds


In [15]:
start_t = time.time()

# ── Ensure threshold_fn is defined (if not already) ──
if 'threshold_fn' not in globals():
    threshold_fn = lambda fc: np.where(
        fc >= np.percentile(fc[np.triu_indices_from(fc, k=1)], 90),
        fc, 0
    )

# ── Helper to build graph with node features ──
def build_graph_with_node_feat(fc_matrix, y, site, subj, node_feat_type='connectivity'):
    adj = threshold_fn(fc_matrix.copy())
    np.fill_diagonal(adj, 0)
    adj_t = torch.tensor(adj, dtype=torch.float32)
    edge_index, edge_attr = dense_to_sparse(adj_t)
    edge_index, edge_attr_sl = add_self_loops(edge_index, edge_attr, num_nodes=fc_matrix.shape[0])
    # Node features
    if node_feat_type == 'identity':
        x = torch.eye(fc_matrix.shape[0], dtype=torch.float32)
    elif node_feat_type == 'connectivity':
        fc_clean = fc_matrix.copy()
        np.fill_diagonal(fc_clean, 0)   # remove diagonal
        x = torch.tensor(fc_clean, dtype=torch.float32)
    elif node_feat_type == 'strength':
        fc_clean = fc_matrix.copy()
        np.fill_diagonal(fc_clean, 0)
        strength = fc_clean.sum(axis=1)   # node strength (sum of absolute or raw?)
        # Use absolute strength to keep non‑negative
        strength = np.abs(strength)
        x = torch.tensor(strength[:, None], dtype=torch.float32)   # shape (n_nodes, 1)
    else:
        raise ValueError(f"Unknown node_feat_type: {node_feat_type}")
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr_sl,
                y=torch.tensor(y, dtype=torch.long),
                site=site, subject=subj)

# ── Lightweight GCN that uses edge weights ──
class WeightedGCN(nn.Module):
    def __init__(self, in_dim, hid=32):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hid)
        self.conv2 = GCNConv(hid, 2)
        self.drop = nn.Dropout(0.3)
    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index, edge_weight=data.edge_attr))
        x = self.drop(x)
        x = self.conv2(x, data.edge_index, edge_weight=data.edge_attr)
        x = global_mean_pool(x, data.batch)
        return x

# ── Quick evaluation using LOSO (Leave-One-Site-Out) ──
def quick_eval_node_feat(node_feat_type, n_epochs=30):
    data_list = [build_graph_with_node_feat(X_fc_norm[i], int(y_binary[i]), sites[i], subjects[i], node_feat_type)
                 for i in range(n_sub)]
    logo = LeaveOneGroupOut()
    fold_aucs = []
    # We'll run only the first 3 sites for speed (or all if you want)
    # For a quick sanity check, we can run all 7, but 3 is enough to see a trend.
    for tr_idx, te_idx in logo.split(np.zeros(n_sub), y_binary, groups=sites):
        # Use only 3 folds for speed; you can loop all sites if needed
        # Here we run all; adjust if too slow.
        train_loader = torch_geometric.loader.DataLoader([data_list[i] for i in tr_idx], batch_size=32, shuffle=True)
        test_loader = torch_geometric.loader.DataLoader([data_list[i] for i in te_idx], batch_size=64)
        in_dim = data_list[0].x.shape[1]
        model = WeightedGCN(in_dim).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
        model.train()
        for _ in range(n_epochs):
            for batch in train_loader:
                batch = batch.to(DEVICE)
                opt.zero_grad()
                out = model(batch)
                loss = F.cross_entropy(out, batch.y)
                loss.backward()
                opt.step()
        model.eval()
        all_probs, all_y = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(DEVICE)
                probs = F.softmax(model(batch), dim=1)[:, 1].cpu().numpy()
                all_probs.extend(probs)
                all_y.extend(batch.y.cpu().numpy())
        try:
            fold_aucs.append(roc_auc_score(all_y, all_probs))
        except ValueError:
            fold_aucs.append(0.5)
    return np.mean(fold_aucs)

# ── Test node features ──
node_feat_types = ['identity', 'connectivity', 'strength']
results = []
for nf in node_feat_types:
    print(f"Evaluating node feature: {nf} ...")
    auc = quick_eval_node_feat(nf, n_epochs=30)
    results.append({'Node_Features': nf, 'AUC_LOSO': round(auc, 4)})
    print(f"  {nf}: AUC = {auc:.4f}")

nf_df = pd.DataFrame(results)
nf_df.to_csv(RES_DIR / 'node_feature_experiment.csv', index=False)

# Select best node feature (by AUC)
selected_node_feat = nf_df.sort_values('AUC_LOSO', ascending=False).iloc[0]['Node_Features']
print(f"\nSelected node features: {selected_node_feat}")

# Save selection info
with open(RES_DIR / 'selected_node_features.json', 'w') as f:
    json.dump({'selected_node_features': selected_node_feat}, f, indent=2)

try:
    log_time('Node Feature Selection', start_t)
except NameError:
    print(f'Node feature selection runtime: {time.time() - start_t:.2f} seconds')

Evaluating node feature: identity ...
  identity: AUC = 0.5350
Evaluating node feature: connectivity ...
  connectivity: AUC = 0.5497
Evaluating node feature: strength ...
  strength: AUC = 0.5227

Selected node features: connectivity
Node feature selection runtime: 1222.81 seconds


In [19]:
start_t = time.time()

# ── Ensure threshold is defined (hard‑coded from prior optimisation) ──
if 'selected_threshold' not in globals():
    selected_threshold = 'top_10pct'
    print("Using default threshold: top_10pct")

# ── Ensure node feature type is defined (load from saved JSON or default) ──
if 'selected_node_feat' not in globals():
    try:
        with open(RES_DIR / 'selected_node_features.json', 'r') as f:
            node_config = json.load(f)
        selected_node_feat = node_config['selected_node_features']
        print(f"Loaded node features: {selected_node_feat}")
    except FileNotFoundError:
        selected_node_feat = 'connectivity'
        print("Using default node features: connectivity")

# ── Ensure threshold function is defined ──
if 'threshold_fn' not in globals():
    threshold_fn = lambda fc: np.where(
        fc >= np.percentile(fc[np.triu_indices_from(fc, k=1)], 90),
        fc, 0
    )

# ── Ensure build_pyg_data is defined (if not, define it) ──
if 'build_pyg_data' not in globals():
    def build_pyg_data(fc_matrix, y, site, subj, node_feat_type='connectivity'):
        assert np.isfinite(fc_matrix).all(), "FC matrix contains non-finite values"
        adj = threshold_fn(fc_matrix.copy())
        np.fill_diagonal(adj, 0)
        assert np.allclose(adj, adj.T), "Adjacency not symmetric"
        adj_t = torch.tensor(adj, dtype=torch.float32)
        edge_index, edge_attr = dense_to_sparse(adj_t)
        assert edge_index.shape[1] > 0, "Thresholded graph has no edges"
        edge_index, edge_attr_sl = add_self_loops(edge_index, edge_attr, num_nodes=fc_matrix.shape[0])
        if node_feat_type == 'identity':
            x = torch.eye(fc_matrix.shape[0], dtype=torch.float32)
        elif node_feat_type == 'connectivity':
            fc_clean = fc_matrix.copy()
            np.fill_diagonal(fc_clean, 0)
            x = torch.tensor(fc_clean, dtype=torch.float32)
        else:
            x = torch.eye(fc_matrix.shape[0], dtype=torch.float32)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr_sl,
                    y=torch.tensor(y, dtype=torch.long),
                    site=site, subject=subj)

# ── Build dataset ──
dataset = [build_pyg_data(X_fc_norm[i], int(y_binary[i]), sites[i], subjects[i], selected_node_feat)
           for i in range(n_sub)]

# ── Full validation for every graph ──
for i, d in enumerate(dataset):
    assert d.x.shape[0] == n_roi, f"Graph {i}: node count mismatch"
    assert d.edge_index.shape[0] == 2, f"Graph {i}: edge_index dims wrong"
    assert torch.isfinite(d.x).all(), f"Graph {i}: NaN in node features"
    assert torch.isfinite(d.edge_attr).all(), f"Graph {i}: NaN in edge weights"
    assert d.edge_index.max() < n_roi, f"Graph {i}: edge index out of range"
    assert d.edge_index.min() >= 0, f"Graph {i}: negative edge index"
    assert d.y.item() in [0, 1], f"Graph {i}: invalid label"

# ── Compute edge counts and undirected density ──
edge_counts = np.array([d.edge_index.shape[1] for d in dataset])
possible_edges = n_roi * (n_roi - 1) / 2          # total possible undirected edges
# Corrected density: remove self‑loops, divide by 2 for undirected, then divide by possible_edges
undirected_edges_per_graph = (edge_counts - n_roi) / 2
density = undirected_edges_per_graph / possible_edges

# ── Summary statistics ──
sample = dataset[0]
preproc_summary = {
    'N_Subjects': n_sub,
    'N_ROI': n_roi,
    'Node_Feature_Dim': int(sample.x.shape[1]),
    'Threshold': selected_threshold,
    'Node_Feature_Type': selected_node_feat,
    'Mean_Edges': int(np.mean(edge_counts)),
    'Std_Edges': int(np.std(edge_counts)),
    'Mean_Density': float(np.mean(density)),        
    'Std_Density': float(np.std(density)),
    'ADHD_Ratio': float(np.mean(y_binary)),
    'N_Sites': int(len(np.unique(sites)))
}
pd.DataFrame([preproc_summary]).to_csv(RES_DIR / 'graph_preprocessing_summary.csv', index=False)
print(json.dumps(preproc_summary, indent=2))

# ── Save full preprocessing configuration as JSON ──
config = {
    "threshold": selected_threshold,
    "node_features": selected_node_feat,
    "n_roi": int(n_roi),
    "n_subjects": int(n_sub),
    "n_sites": int(len(np.unique(sites))),
    "weighted_edges": True,
    "self_loops": True,
    "adhd_ratio": float(np.mean(y_binary)),
    "mean_density": float(np.mean(density)),
    "std_density": float(np.std(density))
}
with open(RES_DIR / 'graph_preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# ── Print one graph summary for debugging ──
print("\nSample graph:")
print(dataset[0])

# ── Runtime logging ──
try:
    log_time('Graph Dataset Construction', start_t)
except NameError:
    print(f'Graph Dataset Construction runtime: {time.time() - start_t:.2f} seconds')

{
  "N_Subjects": 497,
  "N_ROI": 190,
  "Node_Feature_Dim": 190,
  "Threshold": "top_10pct",
  "Node_Feature_Type": "connectivity",
  "Mean_Edges": 3782,
  "Std_Edges": 0,
  "Mean_Density": 0.10002784739626842,
  "Std_Density": 2.7755575615628914e-17,
  "ADHD_Ratio": 0.4346076458752515,
  "N_Sites": 7
}

Sample graph:
Data(x=[190, 190], edge_index=[2, 3782], edge_attr=[3782], y=0, site='KKI', subject='9922944')
Graph Dataset Construction runtime: 155.25 seconds


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
from torch_geometric.nn import global_mean_pool, global_max_pool, global_add_pool

class BaselineGNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, n_classes=2, conv_type='GCN',
                 n_layers=2, dropout=0.5, pool='mean', n_heads=4, use_bn=True):
        super().__init__()
        self.conv_type = conv_type
        self.dropout = dropout
        self.pool_type = pool
        self.n_heads = n_heads
        self.use_bn = use_bn

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList() if use_bn else None

        for i in range(n_layers):
            in_c = in_dim if i == 0 else hidden_dim
            if conv_type == 'GCN':
                self.convs.append(GCNConv(in_c, hidden_dim))
            elif conv_type == 'GAT':
                assert hidden_dim % n_heads == 0, f"hidden_dim ({hidden_dim}) must be divisible by n_heads ({n_heads})"
                out_c = hidden_dim // n_heads
                self.convs.append(GATConv(in_c, out_c, heads=n_heads, concat=True, dropout=dropout))
            elif conv_type == 'SAGE':
                self.convs.append(SAGEConv(in_c, hidden_dim))
            elif conv_type == 'GIN':
                mlp = nn.Sequential(
                    nn.Linear(in_c, hidden_dim),
                    nn.ReLU(),
                    nn.Linear(hidden_dim, hidden_dim)
                )
                self.convs.append(GINConv(mlp))
            else:
                raise ValueError(f"Unsupported conv_type: {conv_type}")

            if use_bn:
                self.bns.append(nn.BatchNorm1d(hidden_dim))

        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, data, return_embedding=False):
        x, edge_index = data.x, data.edge_index
        batch = data.batch
        edge_weight = getattr(data, "edge_attr", None)

        for i, conv in enumerate(self.convs):
            if self.conv_type == 'GCN':
                x = conv(x, edge_index, edge_weight=edge_weight)
            else:
                x = conv(x, edge_index)

            if self.use_bn:
                x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        pool_fn = {
            'mean': global_mean_pool,
            'max': global_max_pool,
            'add': global_add_pool
        }[self.pool_type]
        emb = pool_fn(x, batch)

        if return_embedding:
            return self.classifier(emb), emb
        return self.classifier(emb)

# List of architectures to benchmark
ARCHITECTURES = ['GCN', 'GAT', 'SAGE', 'GIN']
print('Model definitions ready.')

Model definitions ready.


In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import pandas as pd
from torch.cuda.amp import GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score

def compute_class_weights(labels):
    """Compute balanced class weights for cross-entropy loss."""
    labels = labels.astype(int)
    counts = np.bincount(labels)
    if len(counts) == 0:
        raise ValueError("No labels found.")
    weights = len(labels) / (len(counts) * counts.astype(float))
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)

def train_one_epoch(model, loader, optimizer, criterion, scaler=None, clip_grad=1.0):
    """Train one epoch with optional mixed precision and gradient clipping."""
    model.train()
    total_loss = 0.0
    n_samples = 0

    for batch in loader:
        batch = batch.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()

        if scaler is not None:
            with torch.amp.autocast(device_type='cuda', enabled=True):
                out = model(batch)
                if not torch.isfinite(out).all():
                    raise RuntimeError("NaN logits detected in training.")
                loss = criterion(out, batch.y)
                if not torch.isfinite(loss):
                    raise RuntimeError("NaN loss detected in training.")
            scaler.scale(loss).backward()
            if clip_grad > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(batch)
            if not torch.isfinite(out).all():
                raise RuntimeError("NaN logits detected in training.")
            loss = criterion(out, batch.y)
            if not torch.isfinite(loss):
                raise RuntimeError("NaN loss detected in training.")
            loss.backward()
            if clip_grad > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        n_samples += batch.num_graphs

    return total_loss / max(n_samples, 1)

@torch.no_grad()
def evaluate(model, loader, return_embeddings=False):
    """
    Evaluate model on a DataLoader.
    Returns: probs, preds, y_true, embeddings (None if not requested).
    """
    model.eval()
    all_probs = []
    all_preds = []
    all_y = []
    all_embs = [] if return_embeddings else None

    for batch in loader:
        batch = batch.to(DEVICE, non_blocking=True)
        if return_embeddings:
            logits, emb = model(batch, return_embedding=True)
            all_embs.append(emb.cpu().numpy())
        else:
            logits = model(batch)

        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        y = batch.y.cpu().numpy()

        all_probs.append(probs)
        all_preds.append(preds)
        all_y.append(y)

    probs_out = np.concatenate(all_probs)
    preds_out = np.concatenate(all_preds)
    y_out = np.concatenate(all_y)
    if return_embeddings:
        emb_out = np.concatenate(all_embs)
        return probs_out, preds_out, y_out, emb_out
    else:
        return probs_out, preds_out, y_out, None

def train_model(model, train_loader, val_loader, config):
    """
    Train a GNN model with mixed precision, gradient clipping, and early stopping.

    Args:
        model: PyTorch Geometric model
        train_loader: DataLoader for training
        val_loader: DataLoader for validation
        config: dict with keys:
            - lr: learning rate
            - wd: weight decay
            - epochs: max epochs
            - patience: early stopping patience
            - use_amp: bool (default True)
            - clip_grad: float (default 1.0)

    Returns:
        model: trained model (best state restored)
        history: DataFrame with training/validation metrics
        best_epoch: epoch with best validation AUC
        best_val_auc: best validation AUC
        best_val_loss: validation loss at best epoch
    """
    # Mixed precision
    use_amp = config.get('use_amp', True) and torch.cuda.is_available()
    scaler = GradScaler() if use_amp else None

    # Optimizer and scheduler (now tracks validation AUC)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.get('lr', 1e-3),
        weight_decay=config.get('wd', 5e-4)
    )
    scheduler = ReduceLROnPlateau(
        optimizer,
        patience=config.get('scheduler_patience', 5),
        factor=0.5,
        mode='max'            # now tracks AUC (higher is better)
    )

    # Class weights from training labels
    all_labels = np.array([d.y.item() for d in train_loader.dataset])
    cw = compute_class_weights(all_labels)
    criterion = nn.CrossEntropyLoss(weight=cw)

    # Tracking
    best_val_auc = -1.0
    best_val_loss = float('inf')
    best_state = None
    best_epoch = 0
    patience_counter = 0
    history = []

    for epoch in range(config.get('epochs', 150)):
        # Train
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion,
            scaler=scaler,
            clip_grad=config.get('clip_grad', 1.0)
        )

        # Validation (manual loop to compute both loss and AUC)
        model.eval()
        val_loss = 0.0
        n_val = 0
        all_val_probs = []
        all_val_y = []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(DEVICE, non_blocking=True)
                logits = model(batch)
                loss = criterion(logits, batch.y)
                val_loss += loss.item() * batch.num_graphs
                n_val += batch.num_graphs
                probs = F.softmax(logits, dim=1).cpu().numpy()
                all_val_probs.append(probs)
                all_val_y.append(batch.y.cpu().numpy())

        val_loss /= max(n_val, 1)
        val_probs = np.concatenate(all_val_probs)
        val_y = np.concatenate(all_val_y)
        try:
            val_auc = roc_auc_score(val_y, val_probs[:, 1])
        except ValueError:
            val_auc = np.nan

        # Scheduler step (on AUC)
        if not np.isnan(val_auc):
            scheduler.step(val_auc)
        current_lr = optimizer.param_groups[0]['lr']

        # Log
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_auc': val_auc,
            'lr': current_lr
        })

        # Early stopping on validation AUC
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config.get('patience', 15):
                break

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)
    return model, history_df, best_epoch, best_val_auc, best_val_loss

print('Training utilities ready.')

Training utilities ready.


In [33]:
from sklearn.model_selection import StratifiedShuffleSplit
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, recall_score, precision_score, f1_score, brier_score_loss, confusion_matrix

start_t = time.time()

# ── Fixed hyperparameters ──
HIDDEN_DIM = 128
BATCH_SIZE = 64
EPOCHS = 200
PATIENCE = 20
LR = 1e-3
WEIGHT_DECAY = 5e-4
DROPOUT = 0.5
POOL = 'mean'
N_LAYERS = 2

# ── Prepare LOSO folds ──
unique_sites = np.unique(sites)
n_folds = len(unique_sites)
in_dim = dataset[0].x.shape[1]

# Save fold assignments for reproducibility
fold_rows = []
for fold_idx, test_site in enumerate(unique_sites):
    for i in range(n_sub):
        fold_rows.append({
            'Fold': fold_idx,
            'Test_Site': test_site,
            'Subject': subjects[i],
            'Split': 'test' if sites[i] == test_site else 'train'
        })
pd.DataFrame(fold_rows).to_csv(RES_DIR / 'fold_assignments.csv', index=False)

# ── Results containers ──
all_results = []
all_training_curves = []

# ── Loop over architectures ──
for arch in ARCHITECTURES:
    print(f'\n{"="*60}')
    print(f'Architecture: {arch}')
    print('='*60)

    for fold_idx, test_site in enumerate(unique_sites):
        print(f'\n  Fold {fold_idx+1}/{n_folds}: Test site = {test_site}')

        # Indices for this fold
        train_mask = (sites != test_site)
        test_mask = (sites == test_site)
        train_idx = np.where(train_mask)[0]
        test_idx = np.where(test_mask)[0]

        # Skip if test site has only one class
        if len(np.unique(y_binary[test_idx])) < 2:
            print(f'    Skipped (test site has single class)')
            continue

        # ── Split training indices into train/validation (stratified by diagnosis) ──
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
        tr_local, val_local = next(sss.split(train_idx, y_binary[train_idx]))
        train_idx_final = train_idx[tr_local]
        val_idx_final = train_idx[val_local]

        # ── Create DataLoaders from the pre‑built dataset ──
        train_data = [dataset[i] for i in train_idx_final]
        val_data = [dataset[i] for i in val_idx_final]
        test_data = [dataset[i] for i in test_idx]

        train_loader = DataLoader(
            train_data, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=4, pin_memory=True, persistent_workers=True
        )
        val_loader = DataLoader(
            val_data, batch_size=BATCH_SIZE * 2, shuffle=False,
            num_workers=2, pin_memory=True
        )
        test_loader = DataLoader(
            test_data, batch_size=BATCH_SIZE * 2, shuffle=False,
            num_workers=2, pin_memory=True
        )

        # ── Initialize model ──
        model = BaselineGNN(
            in_dim=in_dim,
            hidden_dim=HIDDEN_DIM,
            conv_type=arch,
            n_layers=N_LAYERS,
            dropout=DROPOUT,
            pool=POOL,
            use_bn=True
        ).to(DEVICE)

        # ── Training configuration ──
        config = {
            'lr': LR,
            'wd': WEIGHT_DECAY,
            'epochs': EPOCHS,
            'patience': PATIENCE,
            'use_amp': True,
            'clip_grad': 1.0,
            'scheduler_patience': 5
        }

        # ── Train model (early stopping on validation AUC) ──
        # Now we receive 5 values
        model, history_df, best_epoch, best_val_auc, best_val_loss = train_model(
            model, train_loader, val_loader, config
        )

        # Save training curves
        history_df['architecture'] = arch
        history_df['fold'] = fold_idx
        all_training_curves.append(history_df)

        # ── Save the best model ──
        model_path = MODEL_DIR / f'w2b_{arch}_fold{fold_idx}.pt'
        torch.save(model.state_dict(), model_path)

        # ── Evaluate on held‑out test site ──
        probs, preds, true_y, embs = evaluate(
            model, test_loader, return_embeddings=True
        )
        probs_1 = probs[:, 1] if probs.shape[1] > 1 else probs[:, 0]

        # Metrics
        auc = roc_auc_score(true_y, probs_1) if len(np.unique(true_y)) > 1 else np.nan
        ba = balanced_accuracy_score(true_y, preds)
        sens = recall_score(true_y, preds, pos_label=1, zero_division=0)
        spec = recall_score(true_y, preds, pos_label=0, zero_division=0)
        prec = precision_score(true_y, preds, zero_division=0)
        f1 = f1_score(true_y, preds, zero_division=0)
        brier = brier_score_loss(true_y, probs_1)

        all_results.append({
            'Architecture': arch,
            'Fold': fold_idx,
            'Test_Site': test_site,
            'N_Test': len(test_idx),
            'AUC': auc,
            'BA': ba,
            'Sensitivity': sens,
            'Specificity': spec,
            'Precision': prec,
            'F1': f1,
            'Brier': brier,
            'Best_Epoch': best_epoch,
            'Best_Val_AUC': best_val_auc,
            'Best_Val_Loss': best_val_loss
        })

        # ── Save predictions ──
        test_subjects = subjects[test_idx]
        pred_df = pd.DataFrame({
            'Subject': test_subjects,
            'True_Label': true_y,
            'Pred_Label': preds,
            'Pred_Prob': probs_1
        })
        pred_df.to_csv(
            RES_DIR / 'predictions' / f'w2b_preds_{arch}_fold{fold_idx}.csv',
            index=False
        )

        # ── Save embeddings ──
        np.save(
            RES_DIR / 'embeddings' / f'w2b_emb_{arch}_fold{fold_idx}.npy',
            embs
        )

        # ── Confusion matrix ──
        cm = confusion_matrix(true_y, preds)
        pd.DataFrame(cm).to_csv(
            RES_DIR / 'confusion_matrices' / f'w2b_cm_{arch}_fold{fold_idx}.csv',
            index=False
        )

        print(f'    AUC: {auc:.4f}  BA: {ba:.4f}  Best Epoch: {best_epoch}  Best Val AUC: {best_val_auc:.4f}')

# ── Aggregate results ──
results_df = pd.DataFrame(all_results)
results_df.to_csv(RES_DIR / 'w2b_loso_results.csv', index=False)

# ── Training curves (all folds) ──
all_curves_df = pd.concat(all_training_curves, ignore_index=True)
all_curves_df.to_csv(RES_DIR / 'training_curves.csv', index=False)

# ── Summary per architecture ──
summary = results_df.groupby('Architecture').agg(
    mean_auc=('AUC', 'mean'),
    std_auc=('AUC', 'std'),
    mean_ba=('BA', 'mean'),
    std_ba=('BA', 'std'),
    mean_f1=('F1', 'mean'),
    std_f1=('F1', 'std'),
    n_folds=('Fold', 'count')
).reset_index()
summary.to_csv(RES_DIR / 'architecture_summary.csv', index=False)

print('\n' + '='*60)
print('WORKFLOW 2B COMPLETE')
print('='*60)
print(summary.to_string(index=False))
print(f'\nTotal runtime: {(time.time() - start_t) / 60:.2f} minutes')


Architecture: GCN

  Fold 1/7: Test site = KKI
    AUC: 0.6021  BA: 0.5458  Best Epoch: 2  Best Val AUC: 0.6521

  Fold 2/7: Test site = NYU
    AUC: 0.4653  BA: 0.5328  Best Epoch: 0  Best Val AUC: 0.6358

  Fold 3/7: Test site = NeuroIMAGE
    AUC: 0.5374  BA: 0.5426  Best Epoch: 2  Best Val AUC: 0.6442

  Fold 4/7: Test site = OHSU
    AUC: 0.4929  BA: 0.5119  Best Epoch: 0  Best Val AUC: 0.6811

  Fold 5/7: Test site = Peking_1
    AUC: 0.5478  BA: 0.5318  Best Epoch: 3  Best Val AUC: 0.6859

  Fold 6/7: Test site = Peking_2
    AUC: 0.5027  BA: 0.4996  Best Epoch: 1  Best Val AUC: 0.7089

  Fold 7/7: Test site = Peking_3
    AUC: 0.6796  BA: 0.6201  Best Epoch: 3  Best Val AUC: 0.5897

Architecture: GAT

  Fold 1/7: Test site = KKI
    AUC: 0.6282  BA: 0.5518  Best Epoch: 0  Best Val AUC: 0.6836

  Fold 2/7: Test site = NYU
    AUC: 0.5670  BA: 0.5477  Best Epoch: 12  Best Val AUC: 0.7296

  Fold 3/7: Test site = NeuroIMAGE
    AUC: 0.5548  BA: 0.5609  Best Epoch: 12  Best Val AU

In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.calibration import calibration_curve
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

# ── Set up plotting style ──
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)

# ── Load results ──
results_df = pd.read_csv(RES_DIR / 'w2b_loso_results.csv')
training_curves = pd.read_csv(RES_DIR / 'training_curves.csv')
ARCHITECTURES = results_df['Architecture'].unique()

# ── 1. Summary with confidence intervals ──
def mean_ci(data, conf=0.95):
    n = len(data)
    if n < 2:
        return np.nan, np.nan
    se = np.std(data, ddof=1) / np.sqrt(n)
    ci = se * stats.t.ppf((1 + conf) / 2, n - 1)
    return data.mean() - ci, data.mean() + ci

summary = []
for arch in ARCHITECTURES:
    sub = results_df[results_df['Architecture'] == arch]
    aucs = sub['AUC'].dropna()
    ba = sub['BA'].dropna()
    lo, hi = mean_ci(aucs)
    summary.append({
        'Architecture': arch,
        'Mean_AUC': aucs.mean(),
        'Std_AUC': aucs.std(),
        'CI_Low': lo,
        'CI_High': hi,
        'Mean_BA': ba.mean(),
        'Std_BA': ba.std(),
        'N_Folds': len(aucs)
    })
summary_df = pd.DataFrame(summary)
summary_df.to_csv(RES_DIR / 'w2b_summary_with_ci.csv', index=False)
print('Summary with confidence intervals:')
print(summary_df.to_string(index=False))

# ── 2. Wilcoxon paired tests ──
auc_matrix = results_df.pivot(index='Fold', columns='Architecture', values='AUC')
test_results = []
for i, a1 in enumerate(ARCHITECTURES):
    for a2 in ARCHITECTURES[i+1:]:
        if a1 in auc_matrix and a2 in auc_matrix:
            paired = auc_matrix[[a1, a2]].dropna()
            if len(paired) > 1:
                stat, p = stats.wilcoxon(paired[a1], paired[a2])
                test_results.append({
                    'Arch1': a1, 'Arch2': a2,
                    'p_value': p,
                    'statistic': stat
                })
test_df = pd.DataFrame(test_results)
test_df.to_csv(RES_DIR / 'w2b_wilcoxon_tests.csv', index=False)
print('\nWilcoxon paired tests:')
print(test_df.to_string(index=False))

# ── 3. ROC curves ──
fig, ax = plt.subplots(figsize=(10, 8))
colors = {'GCN': 'blue', 'GAT': 'green', 'SAGE': 'orange', 'GIN': 'red'}
for arch in ARCHITECTURES:
    all_tpr = []
    all_fpr = []
    for fold in results_df[results_df['Architecture'] == arch]['Fold']:
        pred_file = RES_DIR / 'predictions' / f'w2b_preds_{arch}_fold{fold}.csv'
        if pred_file.exists():
            df = pd.read_csv(pred_file)
            y_true = df['True_Label'].values
            y_prob = df['Pred_Prob'].values
            if len(np.unique(y_true)) > 1:
                fpr, tpr, _ = roc_curve(y_true, y_prob)
                ax.plot(fpr, tpr, color=colors.get(arch, 'gray'), alpha=0.3, lw=1)
                all_fpr.append(fpr)
                all_tpr.append(tpr)
    # Mean ROC (interpolate to common FPR grid)
    if all_tpr:
        common_fpr = np.linspace(0, 1, 100)
        mean_tpr = np.mean([np.interp(common_fpr, fpr, tpr) for fpr, tpr in zip(all_fpr, all_tpr)], axis=0)
        ax.plot(common_fpr, mean_tpr, color=colors.get(arch, 'gray'), lw=3,
                label=f'{arch} (mean AUC = {results_df[results_df["Architecture"]==arch]["AUC"].mean():.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=2)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (per fold light, mean bold)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'w2b_roc_curves.png', dpi=300)
plt.close()
print('ROC curves saved.')

# ── 4. Calibration curves ──
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for idx, arch in enumerate(ARCHITECTURES):
    all_y, all_prob = [], []
    for fold in results_df[results_df['Architecture'] == arch]['Fold']:
        pred_file = RES_DIR / 'predictions' / f'w2b_preds_{arch}_fold{fold}.csv'
        if pred_file.exists():
            df = pd.read_csv(pred_file)
            all_y.extend(df['True_Label'].values)
            all_prob.extend(df['Pred_Prob'].values)
    if len(all_y) > 0:
        prob_true, prob_pred = calibration_curve(all_y, all_prob, n_bins=10)
        axes[idx].plot(prob_pred, prob_true, marker='o', label=arch)
        axes[idx].plot([0, 1], [0, 1], 'k--')
        axes[idx].set_title(f'{arch} (Brier = {results_df[results_df["Architecture"]==arch]["Brier"].mean():.3f})')
        axes[idx].set_xlabel('Predicted Probability')
        axes[idx].set_ylabel('Observed Frequency')
        axes[idx].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'w2b_calibration_curves.png', dpi=300)
plt.close()
print('Calibration curves saved.')

# ── 5. Training curves ──
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for idx, arch in enumerate(ARCHITECTURES):
    arch_curves = training_curves[training_curves['architecture'] == arch]
    if arch_curves.empty:
        continue
    # Average across folds
    grouped = arch_curves.groupby('epoch').agg({
        'train_loss': 'mean',
        'val_loss': 'mean',
        'val_auc': 'mean'
    }).reset_index()
    ax = axes[idx]
    ax.plot(grouped['epoch'], grouped['train_loss'], label='Train Loss')
    ax.plot(grouped['epoch'], grouped['val_loss'], label='Val Loss')
    ax2 = ax.twinx()
    ax2.plot(grouped['epoch'], grouped['val_auc'], 'g--', label='Val AUC')
    ax.set_title(arch)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax2.set_ylabel('AUC')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'w2b_training_curves.png', dpi=300)
plt.close()
print('Training curves saved.')

# ── 6. Per‑site AUC bar chart ──
site_order = ['KKI', 'NYU', 'NeuroIMAGE', 'OHSU', 'Peking_1', 'Peking_2', 'Peking_3']
site_order = [s for s in site_order if s in results_df['Test_Site'].unique()]
fig, ax = plt.subplots(figsize=(12, 6))
for arch in ARCHITECTURES:
    sub = results_df[results_df['Architecture'] == arch]
    # Sort by site order
    sub = sub.set_index('Test_Site').reindex(site_order).reset_index()
    ax.plot(sub['Test_Site'], sub['AUC'], marker='o', label=arch)
ax.axhline(y=0.5, color='gray', linestyle='--')
ax.set_xlabel('Test Site')
ax.set_ylabel('AUC')
ax.set_title('Per‑site AUC by Architecture')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / 'w2b_per_site_auc.png', dpi=300)
plt.close()
print('Per‑site AUC plot saved.')

# ── 7. Average confusion matrices ──
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()
for idx, arch in enumerate(ARCHITECTURES):
    cm_sum = None
    n_folds = 0
    for fold in results_df[results_df['Architecture'] == arch]['Fold']:
        cm_file = RES_DIR / 'confusion_matrices' / f'w2b_cm_{arch}_fold{fold}.csv'
        if cm_file.exists():
            cm = pd.read_csv(cm_file, header=None).values
            if cm_sum is None:
                cm_sum = np.zeros_like(cm, dtype=float)
            cm_sum += cm
            n_folds += 1
    if cm_sum is not None and n_folds > 0:
        cm_avg = cm_sum / n_folds
        sns.heatmap(cm_avg, annot=True, fmt='.2f', cmap='Blues', ax=axes[idx],
                    xticklabels=['Control', 'ADHD'], yticklabels=['Control', 'ADHD'])
        axes[idx].set_title(f'{arch}\n(n={n_folds} folds)')
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('True')
plt.tight_layout()
plt.savefig(FIG_DIR / 'w2b_avg_confusion_matrices.png', dpi=300)
plt.close()
print('Confusion matrices saved.')

print('\nAll analysis artifacts generated in:', RES_DIR)

Summary with confidence intervals:
Architecture  Mean_AUC  Std_AUC   CI_Low  CI_High  Mean_BA   Std_BA  N_Folds
         GCN  0.546838 0.073345 0.479005 0.614670 0.540651 0.038741        7
         GAT  0.575191 0.046604 0.532090 0.618292 0.549030 0.038374        7
        SAGE  0.550201 0.044796 0.508772 0.591630 0.514743 0.036070        7
         GIN  0.543739 0.063314 0.485183 0.602294 0.531283 0.057947        7

Wilcoxon paired tests:
Arch1 Arch2  p_value  statistic
  GCN   GAT 0.156250        5.0
  GCN  SAGE 0.937500       13.0
  GCN   GIN 0.937500       13.0
  GAT  SAGE 0.109375        4.0
  GAT   GIN 0.296875        7.0
 SAGE   GIN 0.687500       11.0
ROC curves saved.
Calibration curves saved.
Training curves saved.
Per‑site AUC plot saved.
Confusion matrices saved.

All analysis artifacts generated in: /mnt/ADHD200_WORKFLOW2_RUN/06_results/workflow2b


In [37]:
import numpy as np
import pandas as pd
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore')

start_t = time.time()

# ── Load results ──
results_df = pd.read_csv(RES_DIR / 'w2b_loso_results.csv')
ARCHITECTURES = results_df['Architecture'].unique()

# ── Summary with t‑based CI (n=7) ──
def mean_ci(data, conf=0.95):
    n = len(data)
    if n < 2:
        return np.nan, np.nan
    se = np.std(data, ddof=1) / np.sqrt(n)
    ci = se * stats.t.ppf((1 + conf) / 2, n - 1)
    return data.mean() - ci, data.mean() + ci

summary_rows = []
for arch in ARCHITECTURES:
    sub = results_df[results_df['Architecture'] == arch]
    aucs = sub['AUC'].dropna().values
    lo, hi = mean_ci(aucs)
    summary_rows.append({
        'Architecture': arch,
        'AUC_Mean': aucs.mean(),
        'AUC_Std': aucs.std(),
        'AUC_CI_Lo': lo,
        'AUC_CI_Hi': hi,
        'BA_Mean': sub['BA'].mean(),
        'BA_Std': sub['BA'].std(),
        'Sens_Mean': sub['Sensitivity'].mean(),
        'Spec_Mean': sub['Specificity'].mean(),
        'Prec_Mean': sub['Precision'].mean(),
        'F1_Mean': sub['F1'].mean(),
        'Brier_Mean': sub['Brier'].mean(),
        'N_Folds': len(aucs)
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RES_DIR / 'w2b_summary_table.csv', index=False)
print('Summary table:')
print(summary_df[['Architecture', 'AUC_Mean', 'AUC_CI_Lo', 'AUC_CI_Hi', 'BA_Mean', 'N_Folds']].to_string(index=False))

# ── Pairwise tests (aligned by fold) ──
auc_pivot = results_df.pivot(index='Fold', columns='Architecture', values='AUC').dropna(axis=0, how='all')

def cohens_dz(x, y):
    diff = x - y
    return np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else np.nan

def cliffs_delta(x, y):
    n = len(x)
    more = sum(1 for i in range(n) if x[i] > y[i])
    less = sum(1 for i in range(n) if x[i] < y[i])
    return (more - less) / n if n > 0 else np.nan

pairwise = []
archs = summary_df['Architecture'].tolist()
for i in range(len(archs)):
    for j in range(i+1, len(archs)):
        a1, a2 = archs[i], archs[j]
        common = auc_pivot[[a1, a2]].dropna()
        if len(common) < 2:
            continue
        x = common[a1].values
        y = common[a2].values
        try:
            stat, pval = stats.wilcoxon(x, y)
        except Exception:
            stat, pval = np.nan, np.nan
        cd = cliffs_delta(x, y)
        dz = cohens_dz(x, y)
        pairwise.append({
            'Architecture_A': a1,
            'Architecture_B': a2,
            'N_Paired': len(common),
            'Wilcoxon_Stat': stat,
            'p_value': pval,
            'Cohen_dz': dz,
            'Cliffs_Delta': cd
        })
pairwise_df = pd.DataFrame(pairwise)
pairwise_df.to_csv(RES_DIR / 'w2b_pairwise_tests.csv', index=False)
print('\nPairwise comparisons:')
print(pairwise_df.to_string(index=False))

# ── Classical ML comparison from W2A ──
try:
    w2a_path = W2A_DIR / 'w2_site_signal_summary.csv'
    if not w2a_path.exists():
        raise FileNotFoundError("W2A summary file not found.")
    w2a_df = pd.read_csv(w2a_path)
    # Find the column that contains AUC values
    auc_col = None
    for col in w2a_df.columns:
        if 'auc' in col.lower() and 'mean' in col.lower():
            auc_col = col
            break
    if auc_col is None:
        # fallback: any column with 'AUC' in name
        for col in w2a_df.columns:
            if 'auc' in col.lower():
                auc_col = col
                break
    if auc_col is None:
        raise ValueError("No AUC column found in W2A results.")
    # Find best model per representation
    best_w2a = w2a_df.loc[w2a_df.groupby('Representation')[auc_col].idxmax()]
    w2a_df.to_csv(RES_DIR / 'w2b_classical_vs_gnn.csv', index=False)
    print('\nClassical ML comparison loaded from W2A.')
    print('Best W2A models:')
    print(best_w2a[['Representation', auc_col]].to_string(index=False))
except (FileNotFoundError, Exception) as e:
    print(f'\nW2A results not available or error: {e}')
    print('Skipping classical ML comparison.')

# ── Select best architecture (balance mean and variance) ──
summary_df['Score'] = summary_df['AUC_Mean'] - 0.5 * summary_df['AUC_Std']
best_arch = summary_df.loc[summary_df['Score'].idxmax(), 'Architecture']
summary_df['Selected'] = summary_df['Architecture'] == best_arch
summary_df.to_csv(RES_DIR / 'w2b_summary_table.csv', index=False)

# ── Save best configuration for Workflow 2C ──
best_config = {
    'architecture': best_arch,
    'hidden_dim': HIDDEN_DIM,
    'n_layers': N_LAYERS,
    'dropout': DROPOUT,
    'pool': POOL,
    'use_bn': True,
    'node_features': selected_node_feat,
    'threshold': selected_threshold,
    'mean_auc': summary_df[summary_df['Architecture'] == best_arch]['AUC_Mean'].values[0],
    'std_auc': summary_df[summary_df['Architecture'] == best_arch]['AUC_Std'].values[0],
}
with open(RES_DIR / 'w2b_best_config.json', 'w') as f:
    json.dump(best_config, f, indent=2)

print(f'\nBest architecture: {best_arch} (score = {summary_df[summary_df["Architecture"] == best_arch]["Score"].values[0]:.4f})')
print(f'Mean AUC = {best_config["mean_auc"]:.4f} ± {best_config["std_auc"]:.4f}')

# ── Runtime ──
print(f'\nAnalysis completed in {time.time() - start_t:.1f} seconds.')

Summary table:
Architecture  AUC_Mean  AUC_CI_Lo  AUC_CI_Hi  BA_Mean  N_Folds
         GCN  0.546838   0.479005   0.614670 0.540651        7
         GAT  0.575191   0.532090   0.618292 0.549030        7
        SAGE  0.550201   0.508772   0.591630 0.514743        7
         GIN  0.543739   0.485183   0.602294 0.531283        7

Pairwise comparisons:
Architecture_A Architecture_B  N_Paired  Wilcoxon_Stat  p_value  Cohen_dz  Cliffs_Delta
           GCN            GAT         7            5.0 0.156250 -0.707762     -0.714286
           GCN           SAGE         7           13.0 0.937500 -0.062570      0.142857
           GCN            GIN         7           13.0 0.937500  0.049696     -0.142857
           GAT           SAGE         7            4.0 0.109375  0.608950      0.428571
           GAT            GIN         7            7.0 0.296875  0.536414      0.142857
          SAGE            GIN         7           11.0 0.687500  0.159943      0.142857

Classical ML comparison loaded

In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
import time

start_t = time.time()

cal_results = []

for arch in ARCHITECTURES:
    # Collect all fold predictions (each subject appears exactly once across LOSO folds)
    all_true, all_prob = [], []
    for fold_idx in range(n_folds):
        fp = RES_DIR / 'predictions' / f'w2b_preds_{arch}_fold{fold_idx}.csv'
        if not fp.exists():
            continue
        df = pd.read_csv(fp)
        all_true.extend(df['True_Label'].values)
        all_prob.extend(df['Pred_Prob'].values)

    if len(all_true) == 0:
        continue

    all_true = np.array(all_true)
    all_prob = np.array(all_prob)
    n_sub = len(all_true)

    # Brier score
    brier = brier_score_loss(all_true, all_prob)

    # Expected Calibration Error (ECE) with proper bin edges
    n_bins = 10
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for b in range(n_bins):
        if b == n_bins - 1:
            # include the upper edge for the last bin
            mask = (all_prob >= bin_boundaries[b]) & (all_prob <= bin_boundaries[b + 1])
        else:
            mask = (all_prob >= bin_boundaries[b]) & (all_prob < bin_boundaries[b + 1])
        if mask.sum() == 0:
            continue
        avg_conf = all_prob[mask].mean()
        avg_acc = all_true[mask].mean()
        ece += (mask.sum() / len(all_true)) * abs(avg_conf - avg_acc)

    cal_results.append({
        'Architecture': arch,
        'Brier': brier,
        'ECE': ece,
        'N_Subjects': n_sub
    })

    # Reliability diagram
    prob_true, prob_pred = calibration_curve(all_true, all_prob, n_bins=10, strategy='uniform')

    # Save the calibration curve values as CSV
    pd.DataFrame({
        'Predicted_Probability': prob_pred,
        'Observed_Frequency': prob_true
    }).to_csv(RES_DIR / f'calibration_curve_{arch}.csv', index=False)

    # Plot
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.plot(prob_pred, prob_true, 'o-', label=arch)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(f'{arch} Reliability Diagram')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.savefig(
        FIG_DIR / 'reliability_diagrams' / f'reliability_{arch}.png',
        dpi=300,
        bbox_inches='tight'
    )
    plt.close(fig)

# Create calibration summary DataFrame and sort by ECE (lower is better)
cal_df = pd.DataFrame(cal_results)
cal_df = cal_df.sort_values('ECE')
cal_df.to_csv(RES_DIR / 'w2b_calibration.csv', index=False)

print('Calibration metrics (sorted by ECE):')
print(cal_df.to_string(index=False))

# Runtime logging
try:
    log_time('Calibration Analysis', start_t)
except NameError:
    print(f'Calibration analysis runtime: {time.time() - start_t:.2f} seconds')

Calibration metrics (sorted by ECE):
Architecture    Brier      ECE  N_Subjects
         GCN 0.250814 0.071926         497
         GAT 0.255255 0.109387         497
        SAGE 0.278396 0.129930         497
         GIN 0.321245 0.227102         497
Calibration analysis runtime: 0.73 seconds


In [40]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
import json
import time

start_t = time.time()

# ── Load the best architecture ──
with open(RES_DIR / 'w2b_best_config.json', 'r') as f:
    best_config = json.load(f)
best_arch = best_config['architecture']

# ── Filter results for the best architecture ──
best_results = results_df[results_df['Architecture'] == best_arch].copy()

# ── Site‑level ADHD prevalence ──
site_adhd_pct = {}
for site in np.unique(sites):
    mask = sites == site
    site_adhd_pct[site] = np.mean(y_binary[mask]) * 100

# ── Build error analysis dataframe ──
error_rows = []
for _, row in best_results.iterrows():
    site = row['Test_Site']
    auc = row['AUC']
    # Categorise performance
    if not np.isnan(auc):
        if auc < 0.55:
            status = 'Poor'
        elif auc < 0.65:
            status = 'Moderate'
        else:
            status = 'Good'
    else:
        status = 'Unknown'

    error_rows.append({
        'Site': site,
        'N': row['N_Test'],
        'ADHD_Pct': site_adhd_pct.get(site, np.nan),
        'AUC': auc,
        'BA': row['BA'],
        'Sensitivity': row['Sensitivity'],
        'Specificity': row['Specificity'],
        'Precision': row['Precision'],
        'F1': row['F1'],
        'Brier': row['Brier'],
        'Performance_Status': status,
        'Failed': auc < 0.5 if not np.isnan(auc) else True,
    })

error_df = pd.DataFrame(error_rows)

# ── Correlations with site size ──
valid = error_df.dropna(subset=['AUC'])
corr_summary = {}
if len(valid) > 2:
    r_pearson, p_pearson = pearsonr(valid['N'], valid['AUC'])
    r_spearman, p_spearman = spearmanr(valid['N'], valid['AUC'])
    corr_summary['site_size_pearson_r'] = r_pearson
    corr_summary['site_size_pearson_p'] = p_pearson
    corr_summary['site_size_spearman_r'] = r_spearman
    corr_summary['site_size_spearman_p'] = p_spearman
    print(f"Site size vs AUC: Pearson r={r_pearson:.3f} (p={p_pearson:.3f}), Spearman rho={r_spearman:.3f} (p={p_spearman:.3f})")

# ── Motion analysis (using all available motion metrics) ──
motion_cols = [
    c for c in pheno_df.columns
    if any(k in c.lower() for k in ['fd', 'dvars', 'motion', 'framewise'])
    and pd.api.types.is_numeric_dtype(pheno_df[c])
]

if motion_cols:
    print(f"\nMotion columns found: {motion_cols}")
    site_motion = {}
    for mc in motion_cols:
        # Mean motion per site
        site_motion[mc] = pheno_df.groupby(sites)[mc].mean().to_dict()
        # Map to error_df
        error_df[f'Mean_{mc}'] = error_df['Site'].map(site_motion[mc])
        # Correlation with AUC
        corr_data = error_df.dropna(subset=['AUC', f'Mean_{mc}'])
        if len(corr_data) > 2:
            r_pear, p_pear = pearsonr(corr_data['AUC'], corr_data[f'Mean_{mc}'])
            r_spear, p_spear = spearmanr(corr_data['AUC'], corr_data[f'Mean_{mc}'])
            corr_summary[f'motion_{mc}_pearson_r'] = r_pear
            corr_summary[f'motion_{mc}_pearson_p'] = p_pear
            corr_summary[f'motion_{mc}_spearman_r'] = r_spear
            corr_summary[f'motion_{mc}_spearman_p'] = p_spear
            print(f"  {mc} vs AUC: Pearson r={r_pear:.3f} (p={p_pear:.3f}), Spearman rho={r_spear:.3f} (p={p_spear:.3f})")
else:
    print("\nNo motion columns found. Skipping motion analysis.")

# ── Site performance summary ──
best_site = error_df.loc[error_df['AUC'].idxmax(), 'Site'] if len(error_df) > 0 else None
worst_site = error_df.loc[error_df['AUC'].idxmin(), 'Site'] if len(error_df) > 0 else None
mean_auc = error_df['AUC'].mean()
median_auc = error_df['AUC'].median()
std_auc = error_df['AUC'].std()
cv_auc = std_auc / mean_auc if mean_auc > 0 else np.nan

print(f"\nBest site: {best_site} (AUC = {error_df.loc[error_df['AUC'].idxmax(), 'AUC']:.3f})")
print(f"Worst site: {worst_site} (AUC = {error_df.loc[error_df['AUC'].idxmin(), 'AUC']:.3f})")
print(f"Mean AUC across sites: {mean_auc:.3f} ± {std_auc:.3f}")
print(f"Coefficient of variation (AUC): {cv_auc:.3f}")

# ── Save error analysis CSV ──
error_df.to_csv(RES_DIR / 'w2b_error_analysis.csv', index=False)

# ── Save a structured JSON summary ──
analysis_summary = {
    'best_architecture': best_arch,
    'site_summary': {
        'best_site': best_site,
        'worst_site': worst_site,
        'mean_auc': mean_auc,
        'median_auc': median_auc,
        'std_auc': std_auc,
        'cv_auc': cv_auc,
        'failed_sites': error_df[error_df['Failed'] == True]['Site'].tolist(),
    },
    'correlations': corr_summary,
}
with open(RES_DIR / 'w2b_error_analysis_summary.json', 'w') as f:
    json.dump(analysis_summary, f, indent=2)

print("\nError analysis complete. Summary saved.")

# ── Runtime ──
try:
    log_time('Error Analysis', start_t)
except NameError:
    print(f'Error analysis runtime: {time.time() - start_t:.2f} seconds')

Site size vs AUC: Pearson r=-0.311 (p=0.497), Spearman rho=0.000 (p=1.000)

No motion columns found. Skipping motion analysis.

Best site: Peking_3 (AUC = 0.648)
Worst site: OHSU (AUC = 0.514)
Mean AUC across sites: 0.575 ± 0.047
Coefficient of variation (AUC): 0.081

Error analysis complete. Summary saved.
Error analysis runtime: 0.01 seconds


In [42]:
import json
import numpy as np

config_full = {
    # Experiment settings
    "architecture": best_arch,
    "node_features": selected_node_feat,
    "threshold": selected_threshold,

    # Model hyperparameters
    "hidden_dim": 128,
    "n_layers": 2,
    "dropout": 0.5,
    "pooling": "mean",
    "learning_rate": 1e-3,
    "weight_decay": 5e-4,
    "batch_size": 32,
    "epochs": 200,
    "patience": 20,

    # Dataset information
    "n_subjects": int(n_sub),
    "n_sites": int(len(np.unique(sites))),
    "n_rois": int(n_roi),
    "loso_folds": int(n_folds),

    "mean_auc": float(best_results["AUC"].mean()),
    "std_auc": float(best_results["AUC"].std()),
    "mean_ba": float(best_results["BA"].mean()),
    "std_ba": float(best_results["BA"].std()),
    "seed": int(SEED),
    **env_manifest
}

with open(RES_DIR / "config.json", "w") as f:
    json.dump(config_full, f, indent=2)

print("Workflow 2B COMPLETE")

print(f"Best Architecture : {best_arch}")
print(f"Threshold         : {selected_threshold}")
print(f"Node Features     : {selected_node_feat}")
print(f"Subjects          : {n_sub}")
print(f"Sites             : {len(np.unique(sites))}")
print(f"Mean AUC          : {best_results['AUC'].mean():.4f}")
print(f"Mean BA           : {best_results['BA'].mean():.4f}")
print(f"Configuration saved to:")
print(RES_DIR / "config.json")

Workflow 2B COMPLETE
Best Architecture : GAT
Threshold         : top_10pct
Node Features     : connectivity
Subjects          : 497
Sites             : 7
Mean AUC          : 0.5752
Mean BA           : 0.5490
Configuration saved to:
/mnt/ADHD200_WORKFLOW2_RUN/06_results/workflow2b/config.json
